# Emotion Detection - End-to-End NLP Deep Learning Project

This notebook walks through the complete pipeline used to build EmotionLens:
data understanding, preprocessing, tokenization, model building (SimpleRNN / LSTM / GRU),
evaluation, visualization, and saving the production model artifacts consumed by the
Flask app (app.py).

All of the logic below is also available as a reusable, importable script in
../train_pipeline.py. Running the cells here reproduces the same results.


In [ ]:
import sys
sys.path.append('..')
from train_pipeline import *

## Phase 1 - Data Understanding

Load the train/validation/test splits, inspect shape, duplicates, missing values, and class balance.

In [ ]:
df = load_data()
df.head()

## Phase 2 - Data Preprocessing

Lowercasing, URL/number/punctuation removal, stopword removal, and lemmatization via a single reusable preprocess_text() function.

In [ ]:
df = run_preprocessing(df)
df[['text', 'clean_text']].head()

## Phase 3 - Target Encoding

Emotion labels are integer-encoded with LabelEncoder and saved as label_encoder.pkl.

In [ ]:
df, label_encoder = encode_labels(df)
dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))

## Phase 4 - Tokenization

Keras Tokenizer with a 12,000-word vocabulary and an out-of-vocabulary token converts cleaned text into padded integer sequences.

In [ ]:
padded, tokenizer, max_len = tokenize_and_pad(df)
print('max_len:', max_len)
padded[:2]

## Phase 5 - Train / Test Split

train_test_split with test_size=0.2, random_state=42, and stratify=y so both partitions preserve the original (imbalanced) class proportions.

In [ ]:
y = df['label'].values
num_classes = len(label_encoder.classes_)
X_train, X_test, y_train, y_test = split_data(padded, y)

## Phase 6 - Model Building

Three architectures - SimpleRNN, LSTM, and GRU - share the same Embedding -> recurrent layer -> Dropout -> Dense -> Softmax design, trained with EarlyStopping.

In [ ]:
models, histories, results, best_model_name = train_all_models(
    X_train, X_test, y_train, y_test, VOCAB_SIZE, max_len, num_classes
)
results

## Phase 7 - Model Evaluation

Accuracy, precision, recall, F1, a full classification report, and a confusion matrix for the best-performing model.

In [ ]:
best_model = models[best_model_name]
eval_result = evaluate_model(best_model, X_test, y_test, label_encoder, best_model_name)
print('Best model:', best_model_name)

## Phase 8 - Visualization

Training curves, model comparison, confusion matrix heatmap, and prediction distribution - all saved to assets/.

In [ ]:
plot_training_curves(histories)
plot_model_comparison(results)
plot_confusion_matrix(eval_result['cm'], label_encoder.classes_, best_model_name)
plot_prediction_distribution(eval_result['y_pred'], label_encoder, best_model_name)

### Training curves
![training curves](assets/training_curves.png)

### Model comparison
![model comparison](assets/model_comparison.png)

### Confusion matrix (best model)
![confusion matrix](assets/confusion_matrix_GRU.png)


## Phase 9 - Model Saving

The best model is saved as emotion_model.keras, alongside the fitted tokenizer, label encoder, and max sequence length - everything app.py needs to serve predictions.

In [ ]:
save_best_model(best_model, best_model_name)

## Phase 10 - Testing on Custom Sentences

20 hand-written sentences are run through the saved model to sanity-check real-world predictions with confidence scores.

In [ ]:
results_custom = test_custom_sentences(best_model, tokenizer, max_len, label_encoder)
import pandas as pd
pd.DataFrame(results_custom)[['sentence', 'predicted_emotion', 'confidence']]

## Summary

| Model | Test Accuracy |
|---|---|
| SimpleRNN | 0.772 |
| LSTM | 0.873 |
| GRU (deployed) | 0.896 |

The GRU model was selected for deployment and is served in real time by the Flask
application in ../app.py. See ../README.md for setup and deployment instructions.
